# SpaceX Falcon 9 First Stage Landing Prediction

## Lab 8: Machine Learning Prediction

SpaceX advertises Falcon 9 rocket launches on its website with a cost of 62 million dollars; other providers cost upward of 165 million dollars each. Much of the savings is because SpaceX can reuse the first stage. Therefore, if we can determine if the first stage will land, we can determine the cost of a launch. In this lab, we create a machine learning pipeline to predict if the first stage will land given the data from the preceding labs.

## Objectives

- Create a column for the class label and standardize the feature data
- Split the data into training and test sets
- Use GridSearchCV (10-fold cross validation) to find the best hyperparameters for Logistic Regression, SVM, Decision Tree and K-Nearest Neighbors
- Compare the test accuracy of all models and pick the best one

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn import preprocessing
from sklearn.model_selection import train_test_split
from sklearn.model_selection import GridSearchCV
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier

### Helper function: confusion matrix

This function plots a confusion matrix for a model's predictions, making it easy to see true positives, true negatives and the different error types.

In [ ]:
def plot_confusion_matrix(y, y_predict):
    """This function plots the confusion matrix."""
    from sklearn.metrics import confusion_matrix

    cm = confusion_matrix(y, y_predict)
    ax = plt.subplot()
    sns.heatmap(cm, annot=True, ax=ax, fmt='d', cmap='Blues')
    ax.set_xlabel('Predicted labels')
    ax.set_ylabel('True labels')
    ax.set_title('Confusion Matrix')
    ax.xaxis.set_ticklabels(['did not land', 'land'])
    ax.yaxis.set_ticklabels(['did not land', 'landed'])
    plt.show()

In [ ]:
URL1 = "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/datasets/dataset_part_2.csv"
data = pd.read_csv(URL1)
data.head()

In [ ]:
URL2 = 'https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/datasets/dataset_part_3.csv'
X = pd.read_csv(URL2)
X.head(100)

### Prepare the label and standardize the features

We create a NumPy array `Y` from the `Class` column of the labeled dataset, and standardize the feature matrix `X` using `StandardScaler` so that each feature has zero mean and unit variance. Standardization helps models like logistic regression, SVM and KNN that are sensitive to feature scales.

In [ ]:
Y = data['Class'].to_numpy()
Y[:10]

In [ ]:
transform = preprocessing.StandardScaler()
X = transform.fit_transform(X)
X[:2]

### Split the data into training and test sets

We split the data with `test_size=0.2` and `random_state=2`, so 80% of the records are used for training and 20% (18 records) are held out for evaluation.

In [ ]:
X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size=0.2, random_state=2)
print('Train set:', X_train.shape, Y_train.shape)
print('Test set:', X_test.shape, Y_test.shape)

### Logistic Regression

We tune the regularization strength `C` with 10-fold cross validation using `lbfgs` as the solver and the `l2` penalty.

In [ ]:
parameters = {'C': [0.01, 0.1, 1],
              'penalty': ['l2'],
              'solver': ['lbfgs']}

lr = LogisticRegression()
logreg_cv = GridSearchCV(lr, parameters, cv=10)
logreg_cv.fit(X_train, Y_train)

print("Tuned hyperparameters (best parameters): ", logreg_cv.best_params_)
print("Best cross-validation accuracy: ", logreg_cv.best_score_)
print("Test set accuracy: ", logreg_cv.score(X_test, Y_test))

In [ ]:
yhat = logreg_cv.predict(X_test)
plot_confusion_matrix(Y_test, yhat)

### Support Vector Machine

For the SVM we search over four kernels (linear, rbf, poly, sigmoid) and logarithmic grids for the regularization parameter `C` and the kernel coefficient `gamma`.

In [ ]:
parameters = {'kernel': ('linear', 'rbf', 'poly', 'sigmoid'),
              'C': np.logspace(-3, 3, 5),
              'gamma': np.logspace(-3, 3, 5)}

svm = SVC()
svm_cv = GridSearchCV(svm, parameters, cv=10)
svm_cv.fit(X_train, Y_train)

print("Tuned hyperparameters (best parameters): ", svm_cv.best_params_)
print("Best cross-validation accuracy: ", svm_cv.best_score_)
print("Test set accuracy: ", svm_cv.score(X_test, Y_test))

In [ ]:
yhat = svm_cv.predict(X_test)
plot_confusion_matrix(Y_test, yhat)

### Decision Tree

For the decision tree we tune the splitting criterion (gini/entropy), the splitter strategy, the maximum depth, the maximum number of features considered at each split, and the minimum samples required at leaf nodes and internal splits.

In [ ]:
parameters = {'criterion': ['gini', 'entropy'],
              'splitter': ['best', 'random'],
              'max_depth': [2 * n for n in range(1, 10)],
              'max_features': ['auto', 'sqrt'],
              'min_samples_leaf': [1, 2, 4],
              'min_samples_split': [2, 5, 10]}

tree = DecisionTreeClassifier()
tree_cv = GridSearchCV(tree, parameters, cv=10)
tree_cv.fit(X_train, Y_train)

print("Tuned hyperparameters (best parameters): ", tree_cv.best_params_)
print("Best cross-validation accuracy: ", tree_cv.best_score_)
print("Test set accuracy: ", tree_cv.score(X_test, Y_test))

In [ ]:
yhat = tree_cv.predict(X_test)
plot_confusion_matrix(Y_test, yhat)

### K-Nearest Neighbors

Finally, for KNN we search over the number of neighbors (1 to 10), the algorithm used to compute the nearest neighbors, and the Minkowski distance power parameter `p` (1 = Manhattan, 2 = Euclidean).

In [ ]:
parameters = {'n_neighbors': [1, 2, 3, 4, 5, 6, 7, 8, 9, 10],
              'algorithm': ['auto', 'ball_tree', 'kd_tree', 'brute'],
              'p': [1, 2]}

KNN = KNeighborsClassifier()
knn_cv = GridSearchCV(KNN, parameters, cv=10)
knn_cv.fit(X_train, Y_train)

print("Tuned hyperparameters (best parameters): ", knn_cv.best_params_)
print("Best cross-validation accuracy: ", knn_cv.best_score_)
print("Test set accuracy: ", knn_cv.score(X_test, Y_test))

In [ ]:
yhat = knn_cv.predict(X_test)
plot_confusion_matrix(Y_test, yhat)

### Compare the models

We summarize the cross-validation accuracy and the held-out test accuracy of every model in a single DataFrame to find the best performer.

In [ ]:
results = pd.DataFrame({
    'Model': ['Logistic Regression', 'SVM', 'Decision Tree', 'KNN'],
    'Best CV Score': [logreg_cv.best_score_, svm_cv.best_score_, tree_cv.best_score_, knn_cv.best_score_],
    'Test Accuracy': [logreg_cv.score(X_test, Y_test), svm_cv.score(X_test, Y_test),
                      tree_cv.score(X_test, Y_test), knn_cv.score(X_test, Y_test)]
})
results

In [ ]:
ax = results.plot.bar(x='Model', y=['Best CV Score', 'Test Accuracy'], figsize=(10, 5))
plt.ylabel('Accuracy')
plt.ylim(0, 1)
plt.title('Model Comparison')
plt.show()

## Conclusion

All four models achieve the same held-out test accuracy of about 0.833 on this small test set (18 samples), but the **Decision Tree** reaches the highest cross-validation accuracy (about 0.889), which is a more robust estimate of generalization performance. Therefore, the decision tree classifier is the best model for this dataset and can be used to predict whether the Falcon 9 first stage will land successfully — and hence to estimate the cost of a launch.